In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

df = pd.read_csv('../data/heart_disease_uci.csv')
df['target'] = (df['num'] > 0).astype(int)
df = df.drop(columns=['id', 'dataset', 'num'])

print(df.shape)
print(df.head())

(920, 14)
   age     sex               cp  trestbps   chol    fbs         restecg  \
0   63    Male   typical angina     145.0  233.0   True  lv hypertrophy   
1   67    Male     asymptomatic     160.0  286.0  False  lv hypertrophy   
2   67    Male     asymptomatic     120.0  229.0  False  lv hypertrophy   
3   37    Male      non-anginal     130.0  250.0  False          normal   
4   41  Female  atypical angina     130.0  204.0  False  lv hypertrophy   

   thalch  exang  oldpeak        slope   ca               thal  target  
0   150.0  False      2.3  downsloping  0.0       fixed defect       0  
1   108.0   True      1.5         flat  3.0             normal       1  
2   129.0   True      2.6         flat  2.0  reversable defect       1  
3   187.0  False      3.5  downsloping  0.0             normal       0  
4   172.0  False      1.4    upsloping  0.0             normal       0  


In [3]:
# Separate our columns by type
num_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']
cat_cols = ['sex', 'cp', 'restecg', 'slope', 'thal']
bool_cols = ['fbs', 'exang']

# First fill missing values in bool columns with the most common value
bool_imputer = SimpleImputer(strategy='most_frequent')
df[bool_cols] = bool_imputer.fit_transform(df[bool_cols])

# Now convert True/False to 1/0
df[bool_cols] = df[bool_cols].astype(int)

print(df[bool_cols].head())
print("\nMissing values remaining:", df[bool_cols].isnull().sum().sum())

   fbs  exang
0    1      0
1    0      1
2    0      1
3    0      0
4    0      0

Missing values remaining: 0


In [4]:
# Fill missing values in numeric columns with the median value
num_imputer = SimpleImputer(strategy='median')
df[num_cols] = num_imputer.fit_transform(df[num_cols])

print(df[num_cols].head())
print("\nMissing values remaining:", df[num_cols].isnull().sum().sum())

    age  trestbps   chol  thalch  oldpeak   ca
0  63.0     145.0  233.0   150.0      2.3  0.0
1  67.0     160.0  286.0   108.0      1.5  3.0
2  67.0     120.0  229.0   129.0      2.6  2.0
3  37.0     130.0  250.0   187.0      3.5  0.0
4  41.0     130.0  204.0   172.0      1.4  0.0

Missing values remaining: 0


In [5]:
# Fill missing values in categorical columns with most frequent value
cat_imputer = SimpleImputer(strategy='most_frequent')
df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

# Convert categorical columns to numbers using one-hot encoding
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(df.shape)
print(df.head())

(920, 19)
    age  trestbps   chol  fbs  thalch  exang  oldpeak   ca  target  sex_Male  \
0  63.0     145.0  233.0    1   150.0      0      2.3  0.0       0      True   
1  67.0     160.0  286.0    0   108.0      1      1.5  3.0       1      True   
2  67.0     120.0  229.0    0   129.0      1      2.6  2.0       1      True   
3  37.0     130.0  250.0    0   187.0      0      3.5  0.0       0      True   
4  41.0     130.0  204.0    0   172.0      0      1.4  0.0       0     False   

   cp_atypical angina  cp_non-anginal  cp_typical angina  restecg_normal  \
0               False           False               True           False   
1               False           False              False           False   
2               False           False              False           False   
3               False            True              False            True   
4                True           False              False           False   

   restecg_st-t abnormality  slope_flat  slope_upslo

In [6]:
# Convert all True/False to 1/0
df = df.astype(int)

print(df.head())
print("\nData types:")
print(df.dtypes)


   age  trestbps  chol  fbs  thalch  exang  oldpeak  ca  target  sex_Male  \
0   63       145   233    1     150      0        2   0       0         1   
1   67       160   286    0     108      1        1   3       1         1   
2   67       120   229    0     129      1        2   2       1         1   
3   37       130   250    0     187      0        3   0       0         1   
4   41       130   204    0     172      0        1   0       0         0   

   cp_atypical angina  cp_non-anginal  cp_typical angina  restecg_normal  \
0                   0               0                  1               0   
1                   0               0                  0               0   
2                   0               0                  0               0   
3                   0               1                  0               1   
4                   1               0                  0               0   

   restecg_st-t abnormality  slope_flat  slope_upsloping  thal_normal  \
0      

In [7]:
# Separate features (X) and target (y)
X = df.drop(columns=['target'])
y = df['target']

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining target distribution:\n{y_train.value_counts()}")
print(f"\nTest target distribution:\n{y_test.value_counts()}")

Training set: (736, 18)
Test set: (184, 18)

Training target distribution:
target
1    407
0    329
Name: count, dtype: int64

Test target distribution:
target
1    102
0     82
Name: count, dtype: int64


In [8]:
# Scale the numeric columns
scaler = StandardScaler()

scale_cols = ['age', 'trestbps', 'chol', 'thalch', 'oldpeak', 'ca']

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print(X_train.head())

          age  trestbps      chol  fbs    thalch  exang   oldpeak        ca  \
640 -0.063147  1.560270 -1.857816    0 -0.627205      1 -0.676447 -0.357920   
743  2.180526 -0.116411 -1.857816    0  0.087627      0 -0.676447 -0.357920   
890 -0.063147 -0.451747  0.382627    0 -0.627205      1  1.315013 -0.357920   
270  0.791586  0.442483  0.050710    0  0.008202      1  0.319283  1.268187   
654  0.257378  1.280823 -1.857816    0 -1.540602      0 -0.676447 -0.357920   

     sex_Male  cp_atypical angina  cp_non-anginal  cp_typical angina  \
640         1                   0               1                  0   
743         1                   0               1                  0   
890         1                   0               0                  0   
270         1                   0               0                  0   
654         1                   0               1                  0   

     restecg_normal  restecg_st-t abnormality  slope_flat  slope_upsloping  \
640           

In [9]:
import os

# Save preprocessed data
os.makedirs('../data/processed', exist_ok=True)

X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("Preprocessed data saved successfully!")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Preprocessed data saved successfully!
X_train shape: (736, 18)
X_test shape: (184, 18)
